In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage # by default your LLM try --> input --> human or AI meeage

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DYn7W4mM4UnHHzpxVc0i5aMHQp5eb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc870-fc69-7632-865b-aff68151e20c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [3]:
# explor the in-memorychat

history = InMemoryChatMessageHistory()

history.messages

[]

In [10]:


# or you can add the conversation by yourself too

history.add_user_message("I have a billing issue")
history.add_ai_message("Undrstood you query but I can thelp you")

history.add_user_message("I have a billing issue")
history.add_ai_message("Undrstood you query but I can thelp you")


In [11]:
history

InMemoryChatMessageHistory(messages=[HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={}), AIMessage(content='Undrstood you query but I can thelp you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={}), AIMessage(content='Undrstood you query but I can thelp you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [4]:
# MessagesPlaceholder
# a placeholder inside your prompt ( chatprompt templates)

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a professional email support agent"),
    MessagesPlaceholder(variable_name="history"), # slot for past message
    ("human","{input}") # current user message

    ])

prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

In [5]:

# prompt = ChatPromptTemplate.from_messages([
#     ("system","You are a professional email support agent"),
#     ('past_message',"{history}") # slot for past message
#     ("human","{input}") # current user message

#     ])

# prompt

In [6]:
past_messages = [
    HumanMessage(content="Billing complaint from John — invoice #1042, overcharged $250."),
    AIMessage(content="Classified: Billing — High Priority. SLA: 2 hours."),
]


formatted = prompt.format_messages(
    history = past_messages,
    input = "Draft an aplology email for John"
)


In [13]:
formatted

[SystemMessage(content='You are a professional email support agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classified: Billing — High Priority. SLA: 2 hours.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Draft an aplology email for John', additional_kwargs={}, response_metadata={})]

In [ ]:
# RunnableWithMessageHistory

# step 1 : defined your propt
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])

# step 2 LCEL
chain = prompt | llm | StrOutputParser()

In [12]:
# step 3
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [13]:
# step 4:
email_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [ ]:
# rahul will login to the system
cfg = {"configurable":{"session_id":"123rahul"}}

r1 = email_assistant.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg
)

In [15]:
r1

'Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: 1042  \nIncorrect Charge: $500  \nCorrect Charge: $250  '

In [16]:
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

In [17]:
r2

"John's complaint falls under the category of Billing."

In [18]:
cfg = {"configurable":{"session_id":"shubham_321"}}
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)
r2

'Please provide the details of the complaint so I can classify it accordingly.'

In [19]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: 1042  \nIncorrect Charge: $500  \nCorrect Charge: $250  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'shubham_321': InMemoryChatMessageHistory(messages=[HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content='Please provide the details of the complaint so I can 

In [20]:
cfg = {"configurable":{"session_id":"123rahul"}}
r2 = email_assistant.invoke(
    {"input": "can you summarize my last question's answer?"},
    config=cfg
)
r2

"John's complaint is categorized as Billing."

In [21]:
cfg = {"configurable":{"session_id":"xyz123"}}
r2 = email_assistant.invoke(
    {"input": "can you summarize my last question's answer?"},
    config=cfg
)

In [22]:
r2

"I’m here to help with your question! However, I don't have access to previous conversations or questions you've asked. Can you please provide me with some details about your last question or the topic you need a summary on?"

In [23]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: 1042  \nIncorrect Charge: $500  \nCorrect Charge: $250  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="can you summarize my last question's answer?", additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint is categorized as Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], inv